# Ablation and Discussion

Проверки чувствительности для курсовой (P1):

1. **ε-greedy** — trade-off exploration / exploitation (E6).
2. **Thompson Sampling priors** — влияние `alpha_prior` / `beta_prior` на regret.
3. **batch_size** — задержка обновления политики на OBD batch-режиме.

In [ ]:
# --- 1. ε-greedy ablation (E6) ---
import itertools

import pandas as pd

from src.bandits.epsilon_greedy import EpsilonGreedyPolicy
from src.environments.bernoulli import BernoulliBanditEnv
from src.experiments.configs import ExperimentConfig
from src.experiments.runner import compare_policies
from src.evaluation.summary import summarize_runs

reward_probs = [0.03, 0.035, 0.04, 0.05, 0.045]
configs = []
for seed, epsilon in itertools.product(range(10), [0.05, 0.1, 0.2]):
    configs.append(
        ExperimentConfig(
            run_id=f"eps{epsilon}_seed{seed}",
            seed=seed,
            horizon=5000,
            policy_factory=lambda n_arms, s, eps=epsilon: EpsilonGreedyPolicy(n_arms=n_arms, epsilon=eps, seed=s),
            environment_factory=lambda s, rp=reward_probs: BernoulliBanditEnv(rp, horizon=5000, seed=s),
        )
    )

eps_results = compare_policies(configs)
eps_summary = summarize_runs(eps_results)
eps_summary.groupby("policy_name")[["final_cumulative_regret", "suboptimal_share"]].mean()

In [ ]:
# --- 2. Thompson Sampling priors ---
from src.bandits.thompson_sampling import ThompsonSamplingPolicy

prior_grid = [(1.0, 1.0), (1.0, 5.0), (5.0, 1.0), (10.0, 10.0)]
ts_configs = []
for seed, (alpha_prior, beta_prior) in itertools.product(range(10), prior_grid):
    ts_configs.append(
        ExperimentConfig(
            run_id=f"ts_a{alpha_prior}_b{beta_prior}_seed{seed}",
            seed=seed,
            horizon=5000,
            policy_factory=lambda n_arms, s, a=alpha_prior, b=beta_prior: ThompsonSamplingPolicy(
                n_arms=n_arms,
                alpha_prior=a,
                beta_prior=b,
                seed=s,
            ),
            environment_factory=lambda s, rp=reward_probs: BernoulliBanditEnv(rp, horizon=5000, seed=s),
        )
    )

ts_results = compare_policies(ts_configs)
ts_summary = summarize_runs(ts_results)
ts_summary["prior"] = ts_summary["run_id"].str.extract(r"ts_a([0-9.]+)_b([0-9.]+)").agg("_".join, axis=1)
ts_summary.groupby("prior")[["final_cumulative_regret", "suboptimal_share"]].mean().sort_values("final_cumulative_regret")

In [ ]:
# --- 3. batch_size on OBD (batch mode) ---
from pathlib import Path

from src.bandits.epsilon_greedy import EpsilonGreedyPolicy
from src.bandits.fixed_ab import FixedABPolicy
from src.environments.logged_clicks import LoggedClicksBanditEnv
from src.pipeline.loader import load_events

events_path = Path("../data/processed/obd_events.csv")
if not events_path.exists():
    events_path = Path("data/processed/obd_events.csv")
events = load_events(events_path)

batch_sizes = [1, 50, 200, 1000]
batch_configs = []
for seed, batch_size in itertools.product(range(5), batch_sizes):
    for policy_name, factory in {
        "fixed_ab": lambda n_arms, s: FixedABPolicy(n_arms=n_arms, seed=s),
        "epsilon_greedy": lambda n_arms, s: EpsilonGreedyPolicy(n_arms=n_arms, epsilon=0.1, seed=s),
    }.items():
        batch_configs.append(
            ExperimentConfig(
                run_id=f"{policy_name}_bs{batch_size}_seed{seed}",
                seed=seed,
                horizon=len(events),
                mode="batch",
                batch_size=batch_size,
                policy_factory=factory,
                environment_factory=lambda s, ev=events: LoggedClicksBanditEnv(events=ev, seed=s),
            )
        )

batch_results = compare_policies(batch_configs)
batch_summary = summarize_runs(batch_results)
batch_summary["batch_size"] = batch_summary["run_id"].str.extract(r"_bs(\d+)_")[0].astype(int)
batch_summary.groupby(["policy_name", "batch_size"])[["final_cumulative_regret", "suboptimal_share"]].mean()

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for name in ["fig_3_6_epsilon_regret.png", "fig_3_7_ts_priors_regret.png", "fig_3_8_batch_size_regret.png"]:
    path = Path("../outputs/figures") / name
    if path.exists():
        print(name)
        display(Image(filename=str(path), width=600))